# Class-name CLIP embeddings — 3D interactive inspection

The 57 per-class CLIP text embeddings behind `challenge_data/class_similarity.npy`
(see `preprocessing/build_class_similarity.py`), projected to 3D so their relative
positions can actually be inspected — rotate/zoom/hover in the plots below.

Two projections, since they trade off different things:
- **UMAP** (cosine metric): optimizes for preserving *local* neighbor structure, so
  visual clusters are more trustworthy, but distances between distant clusters and
  overall global layout are not meaningful.
- **PCA**: a faithful *linear* projection (positions/distances mean what they look
  like), but CLIP embeddings are high-dimensional and anisotropic (see the class-
  similarity design discussion — most pairs sit at cosine ~0.5 regardless of relation),
  so 3 components only capture a modest slice of the variance (printed below).

In [1]:
import json
import sys
import warnings
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "preprocessing" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import plotly.graph_objects as go
import umap
from sklearn.decomposition import PCA

DATA_ROOT = REPO_ROOT / "challenge_data"
embeddings = np.load(DATA_ROOT / "class_embeddings.npy")          # (57, 768), L2-normalized
sim = np.load(DATA_ROOT / "class_similarity.npy")                  # (57, 57) cosine sim
species = sorted(json.loads((REPO_ROOT / "class_names.json").read_text())["species"],
                  key=lambda s: s["class_id"])
assert embeddings.shape[0] == len(species)
print(f"{embeddings.shape[0]} classes, {embeddings.shape[1]}-dim CLIP text embeddings")

57 classes, 768-dim CLIP text embeddings


## Labels, colors, hover text

In [2]:
def taxon_order(entry: dict) -> str:
    if entry["scientific_name"] in ("empty", "motorcycle"):
        return "n/a"
    t = entry.get("taxonomy")
    return t["order"] if t else "n/a"

orders = [taxon_order(s) for s in species]
unique_orders = sorted(set(orders))
# qualitative palette, one color per taxonomic order; empty/motorcycle get a fixed neutral
palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#17becf", "#bcbd22", "#7f7f7f", "#aec7e8", "#ffbb78",
]
order_color = {o: palette[i % len(palette)] for i, o in enumerate(o for o in unique_orders if o != "n/a")}
order_color["n/a"] = "#444444"
colors = [order_color[o] for o in orders]

hover_text = [
    f"<b>{s.get('common_name') or s['scientific_name']}</b><br>"
    f"{s['scientific_name']}<br>"
    f"order: {taxon_order(s)}<br>"
    f"class_id: {s['class_id']}"
    for s in species
]
labels = [s.get("common_name") or s["scientific_name"] for s in species]
print(f"{len(unique_orders)} taxonomic groups: {unique_orders}")

11 taxonomic groups: ['Artiodactyla', 'Carnivora', 'Columbiformes', 'Didelphimorphia', 'Galliformes', 'Perissodactyla', 'Primates', 'Proboscidea', 'Rodentia', 'Tinamiformes', 'n/a']


## UMAP projection (primary view)

`n_neighbors=12` (roughly a fifth of the 57 points — small enough to resolve
genus-level clusters like the two `mazama` brockets or three `equus` species,
large enough not to fragment into singletons), cosine metric to match how the
similarity matrix itself is computed.

In [3]:
reducer = umap.UMAP(n_components=3, n_neighbors=12, min_dist=0.3, metric="cosine", random_state=0)
coords_umap = reducer.fit_transform(embeddings)

fig = go.Figure(data=[go.Scatter3d(
    x=coords_umap[:, 0], y=coords_umap[:, 1], z=coords_umap[:, 2],
    mode="markers+text",
    text=labels,
    textposition="top center",
    textfont=dict(size=8),
    hovertext=hover_text,
    hoverinfo="text",
    marker=dict(size=5, color=colors, opacity=0.85, line=dict(width=0.5, color="white")),
)])
fig.update_layout(
    title="CLIP class-name embeddings — UMAP 3D (cosine)",
    scene=dict(xaxis_title="UMAP-1", yaxis_title="UMAP-2", zaxis_title="UMAP-3"),
    width=950, height=750, margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()

## Nearest-neighbor edges

Draws a line from each class to its single closest neighbor (by raw cosine
similarity, not the UMAP layout) on top of the UMAP positions — a direct check
that "visually close in the plot" tracks "actually close in embedding space".

In [4]:
sim_no_self = sim.copy()
np.fill_diagonal(sim_no_self, -1.0)
nn_idx = sim_no_self.argmax(axis=1)

edge_x, edge_y, edge_z = [], [], []
for i, j in enumerate(nn_idx):
    edge_x += [coords_umap[i, 0], coords_umap[j, 0], None]
    edge_y += [coords_umap[i, 1], coords_umap[j, 1], None]
    edge_z += [coords_umap[i, 2], coords_umap[j, 2], None]

fig2 = go.Figure(data=[
    go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode="lines",
                 line=dict(width=2, color="rgba(120,120,120,0.5)"), hoverinfo="skip"),
    go.Scatter3d(
        x=coords_umap[:, 0], y=coords_umap[:, 1], z=coords_umap[:, 2],
        mode="markers+text", text=labels, textposition="top center", textfont=dict(size=8),
        hovertext=hover_text, hoverinfo="text",
        marker=dict(size=5, color=colors, opacity=0.9, line=dict(width=0.5, color="white")),
    ),
])
fig2.update_layout(
    title="UMAP 3D with nearest-neighbor edges (by raw cosine similarity)",
    scene=dict(xaxis_title="UMAP-1", yaxis_title="UMAP-2", zaxis_title="UMAP-3"),
    width=950, height=750, margin=dict(l=0, r=0, b=0, t=40), showlegend=False,
)
fig2.show()

## PCA projection (variance-honest alternative)

Linear projection — axis distances are real, but capture only the variance
printed below (CLIP embeddings are high-dimensional and anisotropic, so a
handful of components rarely explains much).

In [5]:
pca = PCA(n_components=3, random_state=0)
coords_pca = pca.fit_transform(embeddings)
print("explained variance ratio:", pca.explained_variance_ratio_,
      "  sum:", f"{pca.explained_variance_ratio_.sum():.3f}")

fig3 = go.Figure(data=[go.Scatter3d(
    x=coords_pca[:, 0], y=coords_pca[:, 1], z=coords_pca[:, 2],
    mode="markers+text", text=labels, textposition="top center", textfont=dict(size=8),
    hovertext=hover_text, hoverinfo="text",
    marker=dict(size=5, color=colors, opacity=0.85, line=dict(width=0.5, color="white")),
)])
fig3.update_layout(
    title="CLIP class-name embeddings — PCA 3D",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    width=950, height=750, margin=dict(l=0, r=0, b=0, t=40),
)
fig3.show()

explained variance ratio: [0.13587119 0.06206592 0.05618304]   sum: 0.254
